## 数据集
https://pan.baidu.com/s/1yAw1LVTftuhQGAC1Y9RdYQ?pwd=6666

tokenizer_train.jsonl

In [2]:
import pandas as pd

# 先看下数据集
data_path = '/home/li/datasets/tokenizer_train.jsonl'
data = pd.read_json(data_path, lines=True)
data.head(5)
# 就是{"text":"xxx"}

,text
0,好的。现在请你将这个文本中的所有的逗号都替换成空格。 好的，请稍等一下，现在我会将文本中的所...
1,帮我回答一道历史题目。清朝时期的八旗共有多少旗人？ 清朝时期八旗旗人总数约为200万人左右，...
2,嗯，谢谢你介绍的做法很详细，但我不喜欢吃鸡蛋，有没有其他菜做法能介绍一下？ 当然，你可以试试...
3,请描述一下如何正确规划个人理财。 正确规划个人理财需要以下几个步骤：\n1.了解自己的财务状...
4,描述一下天堂和地狱的生态系统和环境。 天堂和地狱被认为是灵性信仰中关于死后世界的两种不同概念...


In [3]:
import json


# 读取数据集yield返回一个称为生成器对象的迭代器
def read_texts_from_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            yield data['text']

In [4]:
from tokenizers import (
    models,
    pre_tokenizers,
    Tokenizer,
)

# 初始化tokenizer
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

这段代码用于初始化一个基于 BPE（Byte Pair Encoding）算法的分词器，具体解释如下：

1. **导入模块**: 
   ```python
   from tokenizers import (
       models,
       pre_tokenizers,
       Tokenizer,
   )
   ```
   这里从 `tokenizers` 库中导入了三个模块：`models` 模块包含各种分词模型，`pre_tokenizers` 模块包含预分词器（用于在实际分词之前对文本进行初步处理），`Tokenizer` 类用于生成和管理分词器对象。

2. **初始化分词器**:
   ```python
   tokenizer = Tokenizer(models.BPE())
   ```
   这行代码创建了一个 `Tokenizer` 对象，并使用 `BPE()` 初始化其分词模型。BPE 是一种常用的分词算法，特别在 NLP 领域中被广泛用于字节对编码的分词方式。它通过反复合并字节对（或符号对）来生成固定大小的词汇表。

3. **设置预分词器**:
   ```python
   tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
   ```
   这里设置了一个 `ByteLevel` 预分词器来对输入文本进行初步处理。在这种配置下，`ByteLevel` 预分词器会按照字节级别来处理文本。这种处理方式通常用于处理非拉丁字符或需要精细控制字符级别的分词任务。`add_prefix_space=False` 表示在处理某些文本时，不会在文本开头强制添加一个空格。

综上所述，这段代码的主要作用就是初始化一个基于 BPE 的分词器，并配置它在进行实际分词之前，按字节级别来初步处理文本数据。这个配置特别适用于需要高精度字符处理的场景，比如多语言文本处理。

In [5]:
from tokenizers import trainers

# 定义特殊token
special_tokens = ["<unk>", "<s>", "</s>"]

# 设置训练器并添加特殊token
trainer = trainers.BpeTrainer(
    vocab_size=6400,
    special_tokens=special_tokens,  # 确保这三个token被包含
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

这段代码用于初始化一个 BPE（Byte Pair Encoding）训练器，以训练分词器模型，并确保特定的特殊标记被包含在生成的词汇表中。具体解释如下：

1. **导入模块**:
   ```python
   from tokenizers import trainers
   ```
   从 `tokenizers` 库中导入 `trainers` 模块，用于定义和配置分词器的训练器。

2. **定义特殊标记**:
   ```python
   special_tokens = ["<unk>", "<s>", "</s>"]
   ```
   这里定义了一个 Python 列表 `special_tokens`，包含三个字符串：`"<unk>"`（未知标记），`"<s>"`（开始标记），和 `"</s>"`（结束标记）。这三个特殊标记在处理自然语言任务时，通常用于表示未识别的单词、序列的开始和结束。

3. **设置训练器并添加特殊标记**:
   ```python
   trainer = trainers.BpeTrainer(
       vocab_size=6400,
       special_tokens=special_tokens,  # 确保这三个token被包含
       show_progress=True,
       initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
   )
   ```
   这段代码创建了一个 `BpeTrainer` 实例，用于训练 BPE 分词模型。具体参数如下：
   
   - `vocab_size=6400`: 指定生成的词汇表大小为 6400 个词。这定义了模型在训练后将生成的词表的容量。
   
   - `special_tokens=special_tokens`: 指定在生成的词汇表中，确保包含之前定义的 `special_tokens` 列表中的三个特殊标记。

   - `show_progress=True`: 在训练过程中显示进度信息，方便用户了解训练进展。

   - `initial_alphabet=pre_tokenizers.ByteLevel.alphabet()`: 使用 `ByteLevel` 预分词器的字母表来作为训练的初始字母表。这种设定通常用于确保训练过程中文本的字节级处理。

总结来说，这段代码旨在训练一个 BPE 模型，并通过 `BpeTrainer` 配置了词汇表大小、强制包含的特殊标记和其他训练相关参数，以便在自然语言处理任务中有效地使用该分词器。

In [6]:
# 开始训练
from tokenizers import decoders
import os

# 读取文本数据
texts = read_texts_from_jsonl(data_path)

# 训练tokenizer
tokenizer.train_from_iterator(texts, trainer=trainer)

# 设置解码器
tokenizer.decoder = decoders.ByteLevel()

# 检查特殊token的索引
assert tokenizer.token_to_id("<unk>") == 0
assert tokenizer.token_to_id("<s>") == 1
assert tokenizer.token_to_id("</s>") == 2

home_dir = '/home/li/work/projects/myPorjects/learn-to-build-a-llm/minimind'
# 保存tokenizer
tokenizer_dir = home_dir + "/model/minimind_tokenizer"
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer.save(os.path.join(tokenizer_dir, "tokenizer.json"))
tokenizer.model.save(home_dir + "/model/minimind_tokenizer")

# 手动创建配置文件
config = {
    "add_bos_token": False,
    "add_eos_token": False,
    "add_prefix_space": True,
    "added_tokens_decoder": {
        "0": {
            "content": "<unk>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        },
        "1": {
            "content": "<s>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        },
        "2": {
            "content": "</s>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        }
    },
    "additional_special_tokens": [],
    "bos_token": "<s>",
    "clean_up_tokenization_spaces": False,
    "eos_token": "</s>",
    "legacy": True,
    "model_max_length": 1000000000000000019884624838656,
    "pad_token": None,
    "sp_model_kwargs": {},
    "spaces_between_special_tokens": False,
    "tokenizer_class": "PreTrainedTokenizerFast",
    "unk_token": "<unk>",
    "use_default_system_prompt": False,
    "chat_template": "{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] %}{% endif %}{% if system_message is defined %}{{ system_message }}{% endif %}{% for message in messages %}{% set content = message['content'] %}{% if message['role'] == 'user' %}{{ '<s>user\\n' + content + '</s>\\n<s>assistant\\n' }}{% elif message['role'] == 'assistant' %}{{ content + '</s>' + '\\n' }}{% endif %}{% endfor %}"
}

# 保存配置文件
with open(os.path.join(tokenizer_dir, "tokenizer_config.json"), "w", encoding="utf-8") as config_file:
    json.dump(config, config_file, ensure_ascii=False, indent=4)

print("Tokenizer training completed and saved.")




Tokenizer training completed and saved.


In [7]:
# 训练完评估分词器
from transformers import AutoTokenizer

home_dir = '/home/li/work/projects/myPorjects/learn-to-build-a-llm/minimind'
# 加载预训练的tokenizer
tokenizer = AutoTokenizer.from_pretrained(home_dir + "/model/minimind_tokenizer")

messages = [
    {"role": "system", "content": "你是一个优秀的聊天机器人，总是给我正确的回应！"},
    {"role": "user", "content": '你来自哪里？'},
    {"role": "assistant", "content": '我来自地球'}
]
new_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False
)
print(new_prompt)

# 获取实际词汇表长度（包括特殊符号）
actual_vocab_size = len(tokenizer)
print('tokenizer实际词表长度：', actual_vocab_size)

model_inputs = tokenizer(new_prompt)
print('encoder长度：', len(model_inputs['input_ids']))

input_ids = model_inputs['input_ids']
response = tokenizer.decode(input_ids)
print('decoder和原始文本是否一致：', response == new_prompt)

你是一个优秀的聊天机器人，总是给我正确的回应！<s>user
你来自哪里？</s>
<s>assistant
我来自地球</s>

tokenizer实际词表长度： 6400
encoder长度： 36
decoder和原始文本是否一致： True
